In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import polars as pl
import plotly.express as px
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate


from utilsforecast.losses import *

from utilsforecast.losses import *

import plotly.io as pio

from utilsforecast.losses import *
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial
from sklearn.linear_model import RidgeCV
from utilsforecast.losses import *
from plotting_utils import (
    plot_data_availability_heatmap,
    plot_missing_percentage,
)
from statsforecast.models import SklearnModel

from statsforecast.models import (
    SeasonalNaive,
    AutoETS,
    MSTL,
)
from xgboost import XGBRegressor

In [ ]:
pio.templates.default = "plotly_white"

# Introduction to Local Forecasting

In this section, we will explore **local forecasting**, a fundamental approach in time series analysis. Local forecasting means building and applying a separate model for each individual time series in our dataset, rather than fitting a single model across all series at once.

---

## What is Local Forecasting?

Imagine you have electricity consumption data from hundreds of households, each with its own unique usage pattern. Instead of assuming all households behave similarly, local forecasting treats each household as a unique case and builds a model specifically for its data.

- **Analogy:** Think of local forecasting like tailoring a suit for each person, rather than making one-size-fits-all clothing.
- **Why use it?** Local models can capture the unique trends, seasonality, and anomalies of each series, often leading to more accurate forecasts for individual cases.

---

## How Does Local Forecasting Work?

1. **Select a Single Time Series:**  
    For each unique identifier (e.g., a household), extract its time series data.

2. **Apply a Model:**  
    Fit a forecasting model (such as ARIMA, Exponential Smoothing, or a machine learning model) to that series.

3. **Repeat:**  
    Repeat the process for every time series in your dataset.

4. **Evaluate Performance:**  
    Assess how well each model predicts future values for its respective series.

---

## Why Is This Important?

- **Personalization:** Local models adapt to the specific characteristics of each series.
- **Benchmarking:** They provide a baseline to compare against more complex global models (which learn from all series together).
- **Interpretability:** Easier to understand and diagnose issues for individual series.

---

## What Will We Do Next?

We will apply the forecasting models we've previously explored to multiple single time series from our dataset. By doing so, we'll see:

- How local models perform across different series
- Which models work best for certain types of series
- The strengths and limitations of the local approach

Throughout this process, we will use the `polars` library for efficient data manipulation, `nixtla`'s forecasting tools for modeling, and `plotly` for interactive visualizations.

---

*Let's dive in and see local forecasting in action!*

In [ ]:
data = pl.read_parquet(
    [
        "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet",
    ]
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [ ]:
data = data.select(
    [
        time_,
        id_,
        target_,
        "Acorn",
        "Acorn_grouped",
        "holidays",
        "visibility",
        "windBearing",
        "temperature",
        "dewPoint",
        "pressure",
        "apparentTemperature",
        "windSpeed",
        "precipType",
        "icon",
        "humidity",
        "summary",
    ]
).explode(
    [
        time_,
        target_,
        "holidays",
        "visibility",
        "windBearing",
        "temperature",
        "dewPoint",
        "pressure",
        "apparentTemperature",
        "windSpeed",
        "precipType",
        "icon",
        "humidity",
        "summary",
    ]
)
data.head()

In [ ]:
print("Number of households", data.get_column("unique_id").n_unique())

## Handling Missing Data in Time Series

Missing data is a common challenge in real-world time series datasets. Before building reliable forecasting models, it's crucial to address these gaps to avoid biased or misleading results.

---

### Why Do We Care About Missing Data?

- **Model Accuracy:** Many forecasting models assume complete data. Missing values can disrupt patterns and reduce accuracy.
- **Interpretability:** Gaps in the data can make it harder to understand trends and seasonality.
- **Downstream Processing:** Some algorithms cannot handle missing values at all.

---

### Step 1: Identifying Missing Values

First, we need to detect where and how much data is missing. In our dataset, missing values in the target column (`y`) indicate periods where energy consumption was not recorded.

---

### Step 2: Dropping Series with Excessive Missingness

If a time series has too many missing values, it may not be useful for modeling. We'll set a threshold (e.g., 20% missing) and remove any series exceeding this limit. This ensures we focus on series with enough data for meaningful analysis.

---

### Step 3: Ensuring Sufficient Data for Validation

To properly validate our forecasting models, we need at least one week of available (non-missing) data at the end of each time series. This allows us to set aside a validation window and assess model performance on unseen data.

---

### Step 4: Imputing Remaining Missing Values (Seasonal Forward Fill)

For the remaining series, we will impute missing values using a **seasonal forward fill** approach. This means that for each missing value, we look back exactly one week (the same day and time in the previous week) and use that value to fill the gap. If that value is also missing, we continue looking back in weekly steps until a non-missing value is found, or use a standard forward fill as a fallback.

**Why seasonal forward fill?**  
Many time series, especially those related to human activity (like energy consumption), exhibit strong weekly patterns. By filling missing values with data from the same time in previous weeks, we preserve these seasonal patterns and avoid introducing artificial trends.

---

### Why This Matters

By carefully handling missing data and ensuring each series has enough data for validation, we set the stage for robust and trustworthy time series modeling.

---

*Next, we'll demonstrate how to identify and handle missing values in our dataset using `polars`, and prepare our data for robust time series modeling and validation. We'll also show how to implement seasonal forward fill imputation for weekly data using Python code.*

In [ ]:
data = data.filter(
    pl.col(target_).is_not_null().sum().truediv(pl.len()).over(id_).ge(0.8)
)
print("Number of households", data.get_column("unique_id").n_unique())

In [ ]:
# ## Filtering Series with a Full Week of Recent Data (No Missing Values in Last 336 Steps)

# For half-hourly data, one week = 7 days * 24 hours * 2 = 336 periods.
validation_window = 336

# Step 1: Sort data by unique_id and timestamp to ensure correct ordering
data_sorted = data.sort([id_, time_])

# Step 2: For each series, check if the last 336 target values are all present (not null)
# We'll use polars' group_by and tail to efficiently select the last 336 rows per series
last_week = (
    data_sorted.group_by(id_, maintain_order=True)
    .tail(validation_window)
    .with_columns(
        [
            # Mark if target is not null
            pl.col(target_).is_not_null().alias("not_null")
        ]
    )
)

# Step 3: For each series, check if all last 336 are not null
valid_series = (
    last_week.group_by(id_)
    .agg(pl.col("not_null").all().alias("full_week"))
    .filter(pl.col("full_week"))
    .get_column(id_)
    .to_list()
)

# Step 4: Filter the main data to keep only these valid series
data = data.filter(pl.col(id_).is_in(valid_series))

print(
    f"Number of households with a complete last week: {data.get_column(id_).n_unique()}"
)

In [ ]:
import random

# ## Selecting a Random Sample of 100 Households for Local Forecasting

# To ensure our analysis is representative and unbiased, we'll randomly select 100 unique households
# from the list of valid series. This helps avoid any unintended patterns that might arise from
# simply taking the first 100 households in the list.


# Set a random seed for reproducibility (so results are consistent each time you run the code)
random.seed(42)

# Randomly sample 100 unique household IDs from the valid_series list
selected_ids = random.sample(valid_series, 100)

# Filter the main data to include only these 100 households
data = data.filter(pl.col(id_).is_in(selected_ids))

print(f"Number of households selected: {data.get_column(id_).n_unique()}")

In [ ]:
id_orders = data.select(pl.col(id_)).unique()

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

In [ ]:
data = data.sort([id_, time_]).with_columns(
    target_col.fill_null(target_col.shift(48 * 7).over(id_))
)

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

In [ ]:
data = data.sort([id_, time_]).with_columns(
    target_col.fill_null(target_col.shift(48).over(id_))
)

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

In [ ]:
data = data.sort([id_, time_]).with_columns(
    target_col.fill_null(target_col.shift(-(48 * 7)).over(id_))
)

In [ ]:
plot_missing_percentage(data, id_orders=id_orders)

In [ ]:
plot_data_availability_heatmap(data, id_orders=id_orders)

# Baseline Time Series Forecasting: Building Our First Models

## Introduction

Before diving into advanced forecasting techniques, it's essential to establish **baseline models**. These simple models serve as reference points, helping us understand whether more complex approaches truly add value. In this section, we'll introduce and implement three foundational time series forecasting methods:

- **Daily Seasonal Naive:** Assumes that today's pattern will repeat tomorrow.
- **Weekly Seasonal Naive:** Assumes that this week's pattern will repeat next week.
- **AutoETS (ANA):** An automated Exponential Smoothing model with Additive Error, No Trend, and Additive Seasonality.

These models are not just academic exercises—they are widely used in industry as benchmarks. If a sophisticated model can't outperform these baselines, it's a sign that we may need to rethink our approach.

---

## Why Start with Baseline Models?

- **Simplicity:** Baseline models are easy to understand and quick to implement.
- **Benchmarking:** They provide a minimum standard of performance. Any advanced model should, at the very least, do better than these.
- **Interpretability:** Their logic is transparent, making it easier to spot data issues or unexpected patterns.

---

## The Models We'll Use

### 1. Daily Seasonal Naive

- **Concept:** Predicts that each value will be the same as the value at the same time yesterday.
- **When is it useful?** When the data has strong daily patterns (e.g., electricity usage peaks every evening).
- **Mathematical Expression:**  
    $$
    \hat{y}_{t+h} = y_{t+h-s}
    $$
    where $s$ is the season length (here, $s=48$ for half-hourly data, since there are 48 half-hours in a day).

### 2. Weekly Seasonal Naive

- **Concept:** Predicts that each value will be the same as the value at the same time last week.
- **When is it useful?** When the data has strong weekly cycles (e.g., higher usage on weekends).
- **Mathematical Expression:**  
    $$
    \hat{y}_{t+h} = y_{t+h-S}
    $$
    where $S$ is the weekly season length (here, $S=48 \times 7 = 336$ for half-hourly data).

### 3. AutoETS (ANA)

- **Concept:** Fits an Exponential Smoothing model with Additive Error, No Trend, and Additive Seasonality. This model automatically learns the best way to combine recent observations and seasonal patterns.
- **Why ANA?**  
    - **A:** Additive Error (errors are added, not multiplied)
    - **N:** No Trend (no long-term upward or downward movement)
    - **A:** Additive Seasonality (seasonal effects are added)
- **Mathematical Expression:**  
    The ANA model can be written as:
    $$
    y_t = l_{t-1} + s_{t-m} + \varepsilon_t
    $$
    where $l_{t-1}$ is the previous level, $s_{t-m}$ is the seasonal component, and $\varepsilon_t$ is the error term.

---

## Our Forecasting Setup

- **Data Frequency:** Half-hourly ($30$ minutes per observation)
- **Forecast Horizon:** $48$ steps ahead (i.e., one day)
- **Cross-Validation:** We'll use rolling windows to estimate how well each model predicts unseen data.

---

In the next section, we'll implement these models using the `nixtla` `statsforecast` library and evaluate their performance. This will give us a solid foundation for comparing more advanced forecasting techniques later on.


In [ ]:
sf = StatsForecast(
    models=[
        SeasonalNaive(season_length=48, alias="DailySeasonalNaive"),
        SeasonalNaive(season_length=48 * 7, alias="WeeklySeasonalNaive"),
        AutoETS(season_length=48, model="ANA", damped=False),
    ],
    freq="30m",
)

y_hat = sf.cross_validation(
    df=data.select([id_, time_, target_]),
    h=48,
    step_size=1,
    n_windows=1,
).drop("cutoff")

In [ ]:
metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
mean_eval_df = evaluate(
    y_hat, metrics=metrics, train_df=data.select([id_, time_, target_]), agg_fn="mean"
)
eval_df = evaluate(y_hat, metrics=metrics, train_df=data.select([id_, time_, target_]))

In [ ]:
fig = px.box(
    eval_df.unpivot(index=[id_, "metric"], variable_name="model"),
    y="metric",  # Metrics on the x-axis
    x="value",  # Error value on the y-axis
    color="model",  # Models as color
    title="Model Performance Across Metrics",
    labels={"value": "Error Value", "metric": "Metric", "model": "Model"},
)
fig.show()

In [ ]:
from mlforecast.lag_transforms import (
    RollingStd,
    SeasonalRollingMean,
    SeasonalRollingStd,
    ExponentiallyWeightedMean,
)

lags = [1, 2, 48, 336]
lag_transforms = {
    1: [
        RollingMean(window_size=3),
        RollingMean(window_size=6),
        RollingMean(window_size=12),
        RollingMean(window_size=48),
        RollingStd(window_size=3),
        RollingStd(window_size=6),
        RollingStd(window_size=12),
        RollingStd(window_size=48),
        ExponentiallyWeightedMean(alpha=0.25),
    ],
    48: [
        RollingMean(window_size=7),
        RollingMean(window_size=14),
        RollingStd(window_size=7),
        RollingStd(window_size=14),
        SeasonalRollingMean(season_length=48, window_size=3),
        SeasonalRollingStd(season_length=48, window_size=3),
    ],
    336: [
        RollingMean(window_size=4),
        RollingMean(window_size=8),
        RollingStd(window_size=4),
        RollingStd(window_size=8),
        SeasonalRollingMean(season_length=336, window_size=3),
        SeasonalRollingStd(season_length=336, window_size=3),
    ],
}

In [ ]:
features = [
    partial(
        fourier, season_length=2 * 24, k=10
    ),  # Daily seasonality (48 observations per day)
    partial(
        fourier, season_length=2 * 24 * 7, k=5
    ),  # Weekly seasonality (336 observations per week)
    partial(
        fourier, season_length=2 * 24 * 365, k=3
    ),  # Annual seasonality (approx. 17520 observations per year)
]
data_fourier, data_futr_fourier = pipeline(
    data.select([id_, time_, target_]),
    features=features,
    freq="30m",
    h=48,  # Horizon for future features
)

In [ ]:
mlf = MLForecast(
    models=[],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
)
data_fourier = mlf.preprocess(data_fourier, static_features=[])

## Why We Use `SklearnModel` with `StatsForecast` for Local ML Models

When working with time series forecasting, it's important to understand the distinction between **global** and **local** modeling approaches—especially when using machine learning (ML) models.

---

### What is a Global Model?

A **global model** is trained on data from all time series in your dataset simultaneously. This means the model tries to learn patterns that are shared across all series, leveraging the collective information to make predictions.

- **Example:** If you have electricity usage data from 100 households, a global model would use all households' data together to train a single ML model.

---

### What is a Local Model?

A **local model** is trained separately for each individual time series. Each household, for example, gets its own dedicated model, which only learns from that household's data.

- **Analogy:** Think of a local model as a personal coach for each household, while a global model is like a single coach for the whole team.

---

### How Does `mlforecast` Work?

The `mlforecast` library is designed to fit **global ML models**. When you use `mlforecast`, it automatically combines all series and fits a single model across them. This is powerful for capturing shared patterns, but it doesn't give each series its own tailored model.

---

### Fitting Local ML Models with `StatsForecast` and `SklearnModel`

To fit **local ML models** (one per series), we use the `SklearnModel` wrapper from the `statsforecast` library. Here's how it works:

- `SklearnModel` allows you to wrap any scikit-learn regressor (like `RidgeCV` or `XGBRegressor`) and use it as a local model within `StatsForecast`.
- When you pass `SklearnModel` instances to `StatsForecast`, it will fit a separate ML model for each time series in your data—just like it does for statistical models.

---

### Why This Matters

- **Consistency:** This approach ensures that our ML models are evaluated in the same local modeling framework as our statistical baselines (like ETS or Seasonal Naive).
- **Fair Comparison:** By fitting one model per series, we can directly compare the performance of local ML models to local statistical models.

---

**In summary:**  
> While `mlforecast` is great for global modeling, to build and evaluate local ML models (one per series), we use `SklearnModel` with `StatsForecast`. This gives each time series its own dedicated machine learning model, aligning with the local forecasting approach we've used throughout this analysis.

In [ ]:
sf = StatsForecast(
    models=[
        SklearnModel(RidgeCV()),
        SklearnModel(
            XGBRegressor(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                random_state=42,
                tree_method="hist",
            )
        ),
    ],
    freq="30m",
)

y_hat = sf.cross_validation(
    df=data_fourier,
    h=48,
    step_size=1,
    n_windows=1,
).drop("cutoff")

In [ ]:
mean_eval_df = pl.concat(
    [
        mean_eval_df,
        evaluate(
            y_hat,
            metrics=metrics,
            train_df=data.select([id_, time_, target_]),
            agg_fn="mean",
        ),
    ],
    how="align",
)
eval_df = pl.concat(
    [
        eval_df,
        evaluate(y_hat, metrics=metrics, train_df=data.select([id_, time_, target_])),
    ],
    how="align",
)

In [ ]:
fig = px.box(
    eval_df.unpivot(index=[id_, "metric"], variable_name="model"),
    y="metric",  # Metrics on the x-axis
    x="value",  # Error value on the y-axis
    color="model",  # Models as color
    title="Model Performance Across Metrics",
    labels={"value": "Error Value", "metric": "Metric", "model": "Model"},
)
fig.update_layout(height=600)
fig.show()

## Model Comparison: Local ML Models Outperform Statistical Baselines

Let's take a closer look at our model evaluation results, focusing on the performance of local machine learning (ML) models—specifically, Ridge Regression (`RidgeCV`) and XGBoost (`XGBRegressor`). By examining the aggregated error metrics in `mean_eval_df`, we see that **XGBoost consistently achieves the lowest error values across most metrics**, including MAE, MSE, RMSE, and MASE.

---

### Why Do Local ML Models (Like XGBoost) Perform So Well?

- **Flexible Pattern Recognition:**  
    Unlike traditional statistical models, ML models like XGBoost can capture complex, nonlinear relationships in the data. This means they can learn subtle patterns and interactions that simpler models might miss.

- **Feature Engineering Power:**  
    By incorporating lagged values, rolling statistics, and Fourier terms (which encode seasonality), our ML models have access to a rich set of features. This allows them to model both short-term fluctuations and long-term seasonal effects.

- **Local Modeling Approach:**  
    By fitting a separate ML model for each household (local modeling), we ensure that each model is tailored to the unique consumption patterns of that household. This personalization often leads to better predictive accuracy.

- **Robustness to Outliers and Missing Data:**  
    Tree-based models like XGBoost are naturally robust to outliers and can handle missing values more gracefully than many statistical models.

---

### What Does This Mean for Our Analysis?

- **ML Models Set a New Benchmark:**  
    Since XGBoost outperforms both naive and statistical models (like ETS) on key metrics, it becomes our new baseline for future model development.

- **Importance of Feature Engineering:**  
    The success of ML models here highlights the value of thoughtful feature engineering—especially when dealing with time series data that exhibits both regular seasonality and irregular fluctuations.

- **Suitability for Complex Data:**  
    If your time series data contains nonlinear relationships, interactions, or is influenced by many factors, ML models (with proper features) can provide a significant boost in accuracy.

---

### Key Metrics Comparison (from `mean_eval_df`)

| Metric | Daily Seasonal Naive | Weekly Seasonal Naive | AutoETS | RidgeCV | XGBRegressor |
|--------|----------------------|----------------------|---------|---------|--------------|
| MAE    | 0.174                | 0.198                | 0.150   | 0.108   | **0.102**    |
| MSE    | 0.113                | 0.127                | 0.064   | 0.038   | **0.037**    |
| RMSE   | 0.279                | 0.303                | 0.215   | 0.168   | **0.165**    |
| MASE   | 0.906                | 1.034                | 0.783   | 0.573   | **0.545**    |

*Lower values indicate better performance. XGBoost achieves the lowest error in every metric above.*

---

### Why Does XGBoost Outperform ETS in This Case?

- **ETS (Exponential Smoothing) models are excellent at capturing regular, repeating seasonal patterns.**  
    However, they may struggle with sudden changes, nonlinear effects, or when the relationship between features and the target is complex.

- **XGBoost, with its ability to model nonlinearities and interactions, can adapt to a wider variety of patterns—especially when provided with engineered features that encode seasonality and recent trends.**

---

### Next Steps

- **Continue Feature Exploration:**  
    Try adding new features (e.g., weather variables, holiday indicators) to see if ML models can leverage them for even better forecasts.

- **Experiment with Global Models:**  
    While we've focused on local models (one per household), global models (trained across all households) can sometimes outperform local models, especially when individual series are short or noisy.

- **Model Interpretability:**  
    Use tools like SHAP values to understand which features are most important for the ML models' predictions.

---

**In summary:**  
> By leveraging local ML models like XGBoost, we can achieve highly accurate forecasts that outperform traditional statistical baselines—especially when we thoughtfully engineer features that capture the unique patterns in our time series data. This demonstrates the power of combining modern machine learning techniques with domain knowledge in time series analysis.